## Scenario generation
This notebook guides you through the process of generating a scenario for Google Earth Studio, in a `.esp` format, corresponding to positions in a specific landing cone.

Before generating a scenario, the trajectory parameters can be defined in 2 different ways presented below:
1) By modifying a `Config` object as illustrated in this notebook
2) Using a scenario file `.yaml`, (equivalent to the `CLI` method presented in the [Readme](README.md))

In [ ]:
import json
import os
from pathlib import Path

from src.scenario.write_scenario import write_scenario
from src.scenario.scenario_config import ScenarioConfig

## Scenario generation using a `Config` object

- Visualisation of the available airports in the dabatase

In [ ]:
data_file = './data/runways_db_V2_FLSim.json'

with open(data_file, 'r') as f:
    runways_database = json.load(f)
airports = [airport for airport in runways_database]

print(airports)

- Once the airports of interest are selected, we can create the corresponding scenarios:

In [ ]:
output_directory = Path("scenarios/TestingAirports") # Creation of the scenario output directory if needed
os.makedirs(output_directory, exist_ok=True)

# The list of airports & runways for which we want to generate a scenario
airports_runways = {
    "EDDF": ["25C", "07C"], # Specific runways of this airport
    "LEMD": [],             # ALL available runways for this airport
}

conf = ScenarioConfig(airports_runways, scenario_dir=output_directory, runways_database_file=data_file)

conf.sample_number = 10 # Number of images to generate for each runway
conf.height = 1024      # target image height/width
conf.width = 1024
conf.fov_x = 60.0
conf.fov_y = 60.0
conf.watermark_height = 0 # recommended: 0

# Date and time parameters
conf.month_max = 8
conf.month_min = 4
conf.day_max = 1
conf.day_min = 1
conf.hour_max = 16
conf.hour_min = 10
conf.minute_max = 0
conf.minute_min = 0       

# Distance to runway parameters
conf.max_distance_m  = 6000 # in meters
conf.min_distance_m  = 150 # Min distance to runway LTP

###########################################################
####       WARNING: ODD SECTION       #####################
conf.use_ODD = False 
# IF True, this flag overrides the conf with the 3-segments 
# configuration file data/Lard_v2_Default_ODD.yaml
###########################################################

# Distribution used for the distances from the runway (details in generate_dist in src/geo/geo_dataset)
conf.distrib_param   = 1.7
conf.distribution    = "exp"     

# Horizontal deviation parameters
conf.alpha_h_distrib = 'uniform' # uniform distribution between min and max
conf.alpha_h_min = -3
conf.alpha_h_max = 3

# Vertical deviation parameters
conf.alpha_v_distrib = 'uniform' 
conf.alpha_v_min = -5.2
conf.alpha_v_max = -1.8

# Roll parameters
conf.roll_distrib = 'uniform'
conf.roll_min = -30
conf.roll_max = 30

# Pitch parameters
conf.pitch_distrib = 'uniform'
conf.pitch_min = -15
conf.pitch_max = 5

# Yaw parameters
conf.yaw_distrib = 'uniform'
conf.yaw_min = -24
conf.yaw_max = 24

# Scenario generation
write_scenario(conf)

## Advanced usage: Airport filtering

In [ ]:
# /!\ For filtering, scikit-learn package is needed
# %pip install scikit-learn

# You can use this experimental tooling to filter airports 
# according to specific characteristics that can be found 
# in metadata retrieved from our_airports.com, combined
# with country/continent/icao informations.

from src.tools.airport_filtering import AirportFiltering

# Initialize the AirportFiltering class
af = AirportFiltering(
    airports_file='data/airport_data/fused_airports_ourAirports_REDUCED.csv',
    runways_file='data/airport_data/fused_runways_FLSim_REDUCED.csv',
    runways_database_file='data/runways_db_V2_GEarth.json' # This is the database file to filter
)

# Reset any existing filters
af.reset_filters()

af.filter_by_airport_type(['large_airport']) \
    .filter_by_continent('EU') \
    .filter_by_runway_length(min_length=3000, max_length=4000) \
    .filter_by_number_of_runways(False)

## Additional filters:
# .filter_by_country(['US', 'CA'])
# .filter_by_airport_type(['large_airport', 'medium_airport'])
# .filter_by_region(['US-CA', 'CA-ON'])
# .filter_by_continent('NA')
# .filter_by_icao_prefix(['K', 'C'])
# .filter_by_runway_length(min_length=2000, max_length=4000)
# .filter_by_runway_width(min_width=30, max_width=60)

# List of countries available are shown in data/ICAO_regions.json

print(len(af.filtered_df))

selected_airports = af.select_random_airports(5)
airports_runways = af.get_airports_runways(selected_airports)
print("Selected Airports:")
for airport, runways in airports_runways.items():
    print(f"Airport: {airport}, Runways: {runways}")
